# 05. Data Cleaning & Feature Preparation

## Objectives

- Validate raw dataset quality and integrity.
- Preserve natural class distributions and missing values.
- Prepare IEEE-CIS FullFeatures and NoCardID.
- Normalize BAF variants independently.
- Export processed datasets in Parquet format.
- Generate reproducible feature schemas.
- Prevent data-dependent preprocessing leakage.

In [1]:
from pathlib import Path
import sys
import gc

import numpy as np
import pandas as pd

from IPython.display import display

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("Project root:", PROJECT_ROOT)

Project root: d:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining


### Import Project Modules

In [2]:
from src.data_loader import (
    load_config,
    resolve_path,
    load_ieee_transactions,
    load_baf,
)

from src.data_profiling import (
    dataset_overview,
    duplicate_report,
    target_validation,
    export_report,
)

from src.preprocessing import (
    load_feature_config,
    preprocess_ieee,
    preprocess_baf,
    get_ieee_feature_columns,
    get_baf_feature_columns,
    audit_baf_sentinels,
    build_feature_schema,
    save_feature_schema,
    save_parquet,
)

config = load_config()
feature_config = load_feature_config()

report_dir = resolve_path(config["paths"]["preprocessing_report_dir"])

report_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Configuration loaded successfully.")

Configuration loaded successfully.


### Initialize Report Containers

In [3]:
preprocessing_records = []

feature_schemas = {
    "preprocessing_version": config["preprocessing"]["version"],
    "ieee_cis": {},
    "baf": {},
}

baf_schema_records = []

baf_sentinel_reports = []

## 2. IEEE-CIS Raw Data Validation

The primary benchmark uses transaction data only
to preserve the 391-feature and 381-feature
settings of the pilot experiments.

Identity data is retained in the raw directory
but excluded from the current experimental pipeline.

### Dataset Overview

In [4]:
ieee_raw = load_ieee_transactions()

print("Raw shape:", ieee_raw.shape)

display(ieee_raw.head())

Raw shape: (590540, 394)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
ieee_overview = dataset_overview(
    ieee_raw,
    dataset_name="IEEE-CIS Raw",
)

display(ieee_overview.T)

,0
dataset_name,IEEE-CIS Raw
num_rows,590540
num_columns,394
numeric_columns,380
non_numeric_columns,14
total_missing_values,95566686
missing_rate_pct,41.073431
memory_usage_mb,1791.667439


### Duplicate ID and Label Validation

In [6]:
ieee_duplicate_check = duplicate_report(
    ieee_raw,
    id_column="TransactionID",
    check_full_rows=False,
)

ieee_target_check = target_validation(
    ieee_raw,
    target_column="isFraud",
)

display(ieee_duplicate_check)

display(ieee_target_check)

,total_rows,duplicate_rows,duplicate_id_count,missing_id_count
0,590540,None,0,0


,target_column,total_rows,missing_labels,invalid_labels,fraud_count,legitimate_count,fraud_rate_pct
0,isFraud,590540,0,0,20663,569877,3.499001


### Numerical Quality Checks

In [7]:
ieee_quality_checks = pd.DataFrame(
    [
        {
            "check": "transaction_rows",
            "value": len(ieee_raw),
        },
        {
            "check": "transaction_columns",
            "value": ieee_raw.shape[1],
        },
        {
            "check": "duplicate_ids",
            "value": int(ieee_raw["TransactionID"].duplicated().sum()),
        },
        {
            "check": "missing_ids",
            "value": int(ieee_raw["TransactionID"].isna().sum()),
        },
        {
            "check": "missing_transaction_time",
            "value": int(ieee_raw["TransactionDT"].isna().sum()),
        },
        {
            "check": "minimum_transaction_time",
            "value": ieee_raw["TransactionDT"].min(),
        },
        {
            "check": "maximum_transaction_time",
            "value": ieee_raw["TransactionDT"].max(),
        },
        {
            "check": "negative_transaction_amount",
            "value": int((ieee_raw["TransactionAmt"] < 0).sum()),
        },
        {
            "check": "zero_transaction_amount",
            "value": int((ieee_raw["TransactionAmt"] == 0).sum()),
        },
        {
            "check": "missing_transaction_amount",
            "value": int(ieee_raw["TransactionAmt"].isna().sum()),
        },
        {
            "check": "minimum_transaction_amount",
            "value": ieee_raw["TransactionAmt"].min(),
        },
        {
            "check": "maximum_transaction_amount",
            "value": ieee_raw["TransactionAmt"].max(),
        },
    ]
)

display(ieee_quality_checks)

export_report(
    ieee_quality_checks,
    report_dir / "ieee_quality_checks.csv",
)

,check,value
0,transaction_rows,5.905400e+05
1,transaction_columns,3.940000e+02
2,duplicate_ids,0.000000e+00
3,missing_ids,0.000000e+00
4,missing_transaction_time,0.000000e+00
5,minimum_transaction_time,8.640000e+04
6,maximum_transaction_time,1.581113e+07
7,negative_transaction_amount,0.000000e+00
8,zero_transaction_amount,0.000000e+00
9,missing_transaction_amount,0.000000e+00


Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\preprocessing\ieee_quality_checks.csv


### Transaction Amount Distribution

In [8]:
display(
    ieee_raw["TransactionAmt"].describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.999,
        ]
    )
)

count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
1%            9.244000
25%          43.321000
50%          68.769000
75%         125.000000
95%         445.000000
99%        1104.000000
99.9%      2769.807320
max       31937.391000
Name: TransactionAmt, dtype: float64

## 3. IEEE-CIS Preprocessing

Preprocessing preserves:

- All valid transactions.
- Ground-truth labels.
- Missing-value patterns.
- Natural class imbalance.
- Original feature information.

Transactions are sorted by TransactionDT
and TransactionID.

Two feature sets are generated:
FullFeatures and NoCardID.

### Execute Preprocessing

In [9]:
ieee_full, ieee_nocardid = preprocess_ieee(ieee_raw)

print("FullFeatures shape:", ieee_full.shape)

print("NoCardID shape:", ieee_nocardid.shape)

FullFeatures shape: (590540, 394)
NoCardID shape: (590540, 384)


### Verify Feature Sets

In [10]:
full_feature_columns = get_ieee_feature_columns(
    ieee_full,
    feature_set="FullFeatures",
)

nocardid_feature_columns = get_ieee_feature_columns(
    ieee_nocardid,
    feature_set="NoCardID",
)

removed_columns = [
    column for column in full_feature_columns if column not in nocardid_feature_columns
]

print(
    "FullFeatures:",
    len(full_feature_columns),
)

print(
    "NoCardID:",
    len(nocardid_feature_columns),
)

print(
    "Removed columns:",
    removed_columns,
)

FullFeatures: 391
NoCardID: 381
Removed columns: ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']


### Verify Data Integrity

In [11]:
assert len(ieee_raw) == len(ieee_full)

assert len(ieee_raw) == len(ieee_nocardid)

assert len(full_feature_columns) == 391

assert len(nocardid_feature_columns) == 381

assert set(removed_columns) == set(config["datasets"]["ieee_cis"]["no_card_id_columns"])

assert ieee_full["TransactionID"].is_unique

assert ieee_nocardid["TransactionID"].is_unique

assert ieee_full["TransactionDT"].is_monotonic_increasing

assert ieee_nocardid["TransactionDT"].is_monotonic_increasing

assert ieee_raw["isFraud"].sum() == (ieee_full["isFraud"].sum())

assert ieee_full["isFraud"].sum() == (ieee_nocardid["isFraud"].sum())

metadata_columns = [
    "TransactionID",
    "TransactionDT",
    "isFraud",
]

pd.testing.assert_frame_equal(
    ieee_full[metadata_columns],
    ieee_nocardid[metadata_columns],
)

assert not set(metadata_columns).intersection(full_feature_columns)

assert not set(metadata_columns).intersection(nocardid_feature_columns)

print("All IEEE-CIS integrity checks passed.")

All IEEE-CIS integrity checks passed.


### Check Missing Value Preservation

Xác nhận pipeline không làm mất hoặc tự động điền missing values.

In [12]:
missing_comparison = pd.DataFrame(
    {
        "raw_missing": ieee_raw.isna().sum(),
        "processed_missing": (ieee_full.isna().sum()),
    }
)

missing_comparison["difference"] = (
    missing_comparison["processed_missing"] - missing_comparison["raw_missing"]
)

display(missing_comparison[missing_comparison["difference"] != 0])

assert (missing_comparison["difference"] == 0).all()

print("Missing-value patterns preserved.")

,raw_missing,processed_missing,difference


Missing-value patterns preserved.


### Check Categorical Features

In [13]:
ieee_categorical_columns = feature_config["ieee_cis"]["categorical_features"]

categorical_summary = pd.DataFrame(
    [
        {
            "column": column,
            "dtype": str(ieee_full[column].dtype),
            "unique_values": (ieee_full[column].nunique()),
            "missing_count": (ieee_full[column].isna().sum()),
            "in_nocardid": (column in nocardid_feature_columns),
        }
        for column in ieee_categorical_columns
    ]
)

display(categorical_summary)

,column,dtype,unique_values,missing_count,in_nocardid
0,ProductCD,string,5,0,True
1,card1,string,13553,0,False
2,card2,string,500,8933,False
3,card3,string,114,1565,False
4,card4,string,4,1577,False
5,card5,string,119,4259,False
6,card6,string,4,1571,False
7,addr1,string,332,65706,False
8,addr2,string,74,65706,False
9,P_emaildomain,string,59,94456,False


### Save Processed IEEE-CIS

In [14]:
ieee_config = config["datasets"]["ieee_cis"]

full_path = save_parquet(
    ieee_full,
    ieee_config["processed_full_path"],
)

nocardid_path = save_parquet(
    ieee_nocardid,
    ieee_config["processed_nocardid_path"],
)

print(
    "FullFeatures file size:",
    round(
        full_path.stat().st_size / 1024**2,
        2,
    ),
    "MB",
)

print(
    "NoCardID file size:",
    round(
        nocardid_path.stat().st_size / 1024**2,
        2,
    ),
    "MB",
)

Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\ieee_full.parquet
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\ieee_nocardid.parquet
FullFeatures file size: 75.82 MB
NoCardID file size: 71.96 MB


## 4. IEEE-CIS Feature Schema

Explicit feature schemas are generated to ensure
that every retraining policy uses the same
model input columns.

Metadata and ground-truth labels are excluded
from model feature lists.

### Build Feature Schemas

In [15]:
feature_schemas["ieee_cis"]["FullFeatures"] = build_feature_schema(
    ieee_full,
    full_feature_columns,
    ieee_categorical_columns,
)

feature_schemas["ieee_cis"]["NoCardID"] = build_feature_schema(
    ieee_nocardid,
    nocardid_feature_columns,
    ieee_categorical_columns,
)

for name, schema in feature_schemas["ieee_cis"].items():

    print("Feature set:", name)

    print(
        "Total features:",
        schema["num_features"],
    )

    print(
        "Categorical:",
        len(schema["categorical_features"]),
    )

    print(
        "Numerical:",
        len(schema["numerical_features"]),
    )

    print("-" * 40)

Feature set: FullFeatures
Total features: 391
Categorical: 20
Numerical: 371
----------------------------------------
Feature set: NoCardID
Total features: 381
Categorical: 10
Numerical: 371
----------------------------------------


### Collect Preprocessing Summary

In [16]:
for name, df, columns, path in [
    (
        "IEEE-CIS FullFeatures",
        ieee_full,
        full_feature_columns,
        full_path,
    ),
    (
        "IEEE-CIS NoCardID",
        ieee_nocardid,
        nocardid_feature_columns,
        nocardid_path,
    ),
]:

    preprocessing_records.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "model_features": len(columns),
            "fraud_count": int(df["isFraud"].sum()),
            "fraud_rate_pct": (df["isFraud"].mean() * 100),
            "missing_values": int(df.isna().sum().sum()),
            "file_size_mb": (path.stat().st_size / 1024**2),
            "status": "passed",
        }
    )

display(pd.DataFrame(preprocessing_records))

,dataset,rows,columns,model_features,fraud_count,fraud_rate_pct,missing_values,file_size_mb,status
0,IEEE-CIS FullFeatures,590540,394,391,20663,3.499001,95566686,75.818724,passed
1,IEEE-CIS NoCardID,590540,384,381,20663,3.499001,94869664,71.955681,passed


### Release IEEE-CIS Memory

In [17]:
del ieee_raw
del ieee_full
del ieee_nocardid

gc.collect()

print("IEEE-CIS memory released.")

IEEE-CIS memory released.


## 5. Bank Account Fraud Preprocessing

Each BAF variant is processed independently.

The original monthly time resolution is preserved.
No artificial daily timestamps are generated.

Schema differences and candidate sentinel values
are recorded without automatically removing
features or replacing sentinel values.

### Inspect BAF Schema Differences

In [18]:
baf_config = config["datasets"]["baf"]

baf_column_sets = {}

for variant in baf_config["variants"]:

    path = resolve_path(Path(baf_config["directory"]) / variant)

    columns = pd.read_csv(
        path,
        nrows=0,
    ).columns.tolist()

    baf_column_sets[variant] = set(columns)


base_columns = baf_column_sets["Base.csv"]

for variant, columns in baf_column_sets.items():

    extra_columns = sorted(columns - base_columns)

    missing_columns = sorted(base_columns - columns)

    baf_schema_records.append(
        {
            "variant": variant,
            "raw_columns": len(columns),
            "extra_columns": ", ".join(extra_columns),
            "missing_columns": ", ".join(missing_columns),
        }
    )

baf_schema_comparison = pd.DataFrame(baf_schema_records)

display(baf_schema_comparison)

,variant,raw_columns,extra_columns,missing_columns
0,Base.csv,32,,
1,Variant I.csv,32,,
2,Variant II.csv,32,,
3,Variant III.csv,34,"x1, x2",
4,Variant IV.csv,32,,
5,Variant V.csv,34,"x1, x2",


### Identify Common Feature Schema

In [19]:
common_baf_columns = set.intersection(*baf_column_sets.values())

common_baf_features = sorted(
    common_baf_columns
    - {
        baf_config["target_column"],
        baf_config["time_column"],
    }
)

print(
    "Common raw columns:",
    len(common_baf_columns),
)

print(
    "Common model features:",
    len(common_baf_features),
)

print(common_baf_features)

feature_schemas["baf"]["common_features"] = common_baf_features

Common raw columns: 32
Common model features: 30
['bank_branch_count_8w', 'bank_months_count', 'credit_risk_score', 'current_address_months_count', 'customer_age', 'date_of_birth_distinct_emails_4w', 'days_since_request', 'device_distinct_emails_8w', 'device_fraud_count', 'device_os', 'email_is_free', 'employment_status', 'foreign_request', 'has_other_cards', 'housing_status', 'income', 'intended_balcon_amount', 'keep_alive_session', 'name_email_similarity', 'payment_type', 'phone_home_valid', 'phone_mobile_valid', 'prev_address_months_count', 'proposed_credit_limit', 'session_length_in_minutes', 'source', 'velocity_24h', 'velocity_4w', 'velocity_6h', 'zip_count_4w']


### Process All BAF Variants

In [20]:
baf_categorical_columns = feature_config["baf"]["categorical_features"]

for variant in baf_config["variants"]:

    print("=" * 60)
    print("Processing:", variant)

    # 1. Load one variant.
    baf_raw = load_baf(variant)

    raw_rows = len(baf_raw)

    raw_fraud_count = int(baf_raw["fraud_bool"].sum())

    # 2. Audit candidate sentinel values.
    sentinel_report = audit_baf_sentinels(
        baf_raw,
        variant_name=variant,
    )

    baf_sentinel_reports.append(sentinel_report)

    # 3. Preprocess.
    baf_processed = preprocess_baf(
        baf_raw,
        variant_name=variant,
    )

    # 4. Verify integrity.
    assert len(baf_processed) == raw_rows

    assert baf_processed["record_id"].is_unique

    assert baf_processed["month"].is_monotonic_increasing

    assert int(baf_processed["fraud_bool"].sum()) == raw_fraud_count

    # 5. Identify features.
    baf_features = get_baf_feature_columns(baf_processed)

    # 6. Build feature schema.
    feature_schemas["baf"][variant] = build_feature_schema(
        baf_processed,
        baf_features,
        baf_categorical_columns,
    )

    # 7. Create output file name.
    output_name = Path(variant).stem.lower().replace(" ", "_") + ".parquet"

    output_path = Path(baf_config["processed_directory"]) / output_name

    # 8. Save processed dataset.
    saved_path = save_parquet(
        baf_processed,
        output_path,
    )

    # 9. Collect summary.
    preprocessing_records.append(
        {
            "dataset": f"BAF {variant}",
            "rows": len(baf_processed),
            "columns": baf_processed.shape[1],
            "model_features": len(baf_features),
            "fraud_count": int(baf_processed["fraud_bool"].sum()),
            "fraud_rate_pct": (baf_processed["fraud_bool"].mean() * 100),
            "missing_values": int(baf_processed.isna().sum().sum()),
            "file_size_mb": (saved_path.stat().st_size / 1024**2),
            "status": "passed",
        }
    )

    print(
        "Processed shape:",
        baf_processed.shape,
    )

    print(
        "Model features:",
        len(baf_features),
    )

    # 10. Release memory.
    del baf_raw
    del baf_processed

    gc.collect()

print("=" * 60)
print("All BAF variants processed successfully.")

Processing: Base.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\baf\base.parquet
Processed shape: (1000000, 33)
Model features: 30
Processing: Variant I.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\baf\variant_i.parquet
Processed shape: (1000000, 33)
Model features: 30
Processing: Variant II.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\baf\variant_ii.parquet
Processed shape: (1000000, 33)
Model features: 30
Processing: Variant III.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\baf\variant_iii.parquet
Processed shape: (1000000, 35)
Model features: 32
Processing: Variant IV.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\processed\baf\variant_iv.parquet
Processed shape: (1000000, 33)
Model features: 30
Processing: Variant V.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\data\proc

### Analyze Sentinel Values

In [21]:
baf_sentinel_report = pd.concat(
    baf_sentinel_reports,
    ignore_index=True,
)

display(
    baf_sentinel_report.sort_values(
        "percentage",
        ascending=False,
    )
)

,variant,column,candidate_sentinel,count,percentage
15,Variant III.csv,prev_address_months_count,-1,762005,76.2005
20,Variant IV.csv,prev_address_months_count,-1,761771,76.1771
25,Variant V.csv,prev_address_months_count,-1,761730,76.1730
10,Variant II.csv,prev_address_months_count,-1,759415,75.9415
0,Base.csv,prev_address_months_count,-1,712920,71.2920
5,Variant I.csv,prev_address_months_count,-1,704367,70.4367
7,Variant I.csv,bank_months_count,-1,254905,25.4905
2,Base.csv,bank_months_count,-1,253635,25.3635
17,Variant III.csv,bank_months_count,-1,247242,24.7242
22,Variant IV.csv,bank_months_count,-1,247181,24.7181


## 6. Export Preprocessing Reports

The preprocessing outputs include dataset-level
statistics, explicit feature schemas, BAF schema
differences and sentinel-value audit results.

### Export Summary Reports

In [22]:
preprocessing_summary = pd.DataFrame(preprocessing_records)

display(preprocessing_summary)

export_report(
    preprocessing_summary,
    report_dir / "preprocessing_summary.csv",
)

export_report(
    baf_schema_comparison,
    report_dir / "baf_schema_comparison.csv",
)

export_report(
    baf_sentinel_report,
    report_dir / "baf_sentinel_report.csv",
)

save_feature_schema(
    feature_schemas,
    "reports/preprocessing/feature_schema.json",
)

,dataset,rows,columns,model_features,fraud_count,fraud_rate_pct,missing_values,file_size_mb,status
0,IEEE-CIS FullFeatures,590540,394,391,20663,3.499001,95566686,75.818724,passed
1,IEEE-CIS NoCardID,590540,384,381,20663,3.499001,94869664,71.955681,passed
2,BAF Base.csv,1000000,33,30,11029,1.102900,0,70.331963,passed
3,BAF Variant I.csv,1000000,33,30,11029,1.102900,0,70.270853,passed
4,BAF Variant II.csv,1000000,33,30,11029,1.102900,0,70.321290,passed
5,BAF Variant III.csv,1000000,35,32,11030,1.103000,0,86.146969,passed
6,BAF Variant IV.csv,1000000,33,30,11030,1.103000,0,70.360106,passed
7,BAF Variant V.csv,1000000,35,32,11030,1.103000,0,86.085425,passed


Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\preprocessing\preprocessing_summary.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\preprocessing\baf_schema_comparison.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\preprocessing\baf_sentinel_report.csv
Saved: D:\UniversityCourses\HK7\IE212.R12\Project\fraud-retraining\reports\preprocessing\feature_schema.json


## 7. Final Verification

Verify persisted Parquet datasets to ensure
that the preprocessing outputs are readable,
complete and consistent with their feature schemas.

### Verify Saved Parquet Files

In [23]:
import pyarrow.parquet as pq

processed_dir = resolve_path(config["paths"]["processed_dir"])

parquet_files = sorted(processed_dir.rglob("*.parquet"))

verification_records = []

for path in parquet_files:

    parquet_file = pq.ParquetFile(path)

    metadata = parquet_file.metadata

    verification_records.append(
        {
            "file": str(path.relative_to(PROJECT_ROOT)),
            "rows": metadata.num_rows,
            "columns": metadata.num_columns,
            "row_groups": metadata.num_row_groups,
            "size_mb": round(
                path.stat().st_size / 1024**2,
                2,
            ),
        }
    )

parquet_verification = pd.DataFrame(verification_records)

display(parquet_verification)

,file,rows,columns,row_groups,size_mb
0,data\processed\baf\base.parquet,1000000,33,1,70.33
1,data\processed\baf\variant_i.parquet,1000000,33,1,70.27
2,data\processed\baf\variant_ii.parquet,1000000,33,1,70.32
3,data\processed\baf\variant_iii.parquet,1000000,35,1,86.15
4,data\processed\baf\variant_iv.parquet,1000000,33,1,70.36
5,data\processed\baf\variant_v.parquet,1000000,35,1,86.09
6,data\processed\ieee_full.parquet,590540,394,1,75.82
7,data\processed\ieee_nocardid.parquet,590540,384,1,71.96


### Verify Persisted IEEE-CIS Metadata

In [24]:
metadata_columns = [
    "TransactionID",
    "TransactionDT",
    "isFraud",
]

full_metadata = pd.read_parquet(
    resolve_path(ieee_config["processed_full_path"]),
    columns=metadata_columns,
)

nocardid_metadata = pd.read_parquet(
    resolve_path(ieee_config["processed_nocardid_path"]),
    columns=metadata_columns,
)

pd.testing.assert_frame_equal(
    full_metadata,
    nocardid_metadata,
)

assert len(full_metadata) == 590540

assert full_metadata["TransactionID"].is_unique

assert full_metadata["TransactionDT"].is_monotonic_increasing

assert full_metadata["isFraud"].isin([0, 1]).all()

print("Persisted IEEE-CIS metadata checks passed.")

Persisted IEEE-CIS metadata checks passed.


### Inspect Temporal Day Range

In [25]:
seconds_per_day = ieee_config["seconds_per_day"]

day_preview = full_metadata["TransactionDT"] // seconds_per_day

print("Minimum day:", day_preview.min())

print("Maximum day:", day_preview.max())

print("Unique days:", day_preview.nunique())

display(day_preview.value_counts().sort_index().head(10))

Minimum day: 1
Maximum day: 182
Unique days: 182


TransactionDT
1     5122
2     3730
3     3241
4     4036
5     3964
6     3717
7     3786
8     3999
9     3907
10    3445
Name: count, dtype: int64

### Final Summary

In [26]:
expected_files = {
    "ieee_full.parquet",
    "ieee_nocardid.parquet",
    "base.parquet",
    "variant_i.parquet",
    "variant_ii.parquet",
    "variant_iii.parquet",
    "variant_iv.parquet",
    "variant_v.parquet",
}

actual_files = {path.name for path in parquet_files}

assert expected_files.issubset(actual_files)

assert len(preprocessing_summary) == 8

assert (preprocessing_summary["status"] == "passed").all()

print("=" * 60)

print("STEP 3: DATA PREPROCESSING COMPLETED")

print("IEEE-CIS feature sets: 2")

print("BAF variants: 6")

print("Processed datasets: 8")

print("Ready for Step 4: Temporal Partitioning")

print("=" * 60)

STEP 3: DATA PREPROCESSING COMPLETED
IEEE-CIS feature sets: 2
BAF variants: 6
Processed datasets: 8
Ready for Step 4: Temporal Partitioning
